импорты


In [68]:
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from sklearn.metrics import roc_auc_score

In [69]:
data = pd.read_parquet('data/pairs.parquet', columns=['id1', 'id2', 'target', 'name_1', 'name_2'])

In [70]:
print(data.shape)
data[['name_1', 'name_2']].isna().sum()

(365654, 5)


name_1    0
name_2    0
dtype: int64

In [71]:
data['name_1_clean'] = data['name_1'].str.strip().str.lower()
data['name_2_clean'] = data['name_2'].str.strip().str.lower()

### Метрики похожести названий (rapidfuzz.fuzz)

Все метрики возвращают число от 0 до 100: чем больше, тем названия похожее.

- **`ratio`** — сравнивает строки целиком посимвольно (на основе расстояния Левенштейна: сколько вставок, удалений и замен символов нужно, чтобы превратить одну строку в другую).
- **`partial_ratio`** — ищет самый похожий на короткую строку фрагмент длинной строки и сравнивает с ним. Высокий, если одно название почти целиком содержится в другом.
- **`token_sort_ratio`** — разбивает названия на слова, сортирует их по алфавиту и затем сравнивает как `ratio`. Не зависит от порядка слов.
- **`token_set_ratio`** — сравнивает общие слова двух названий и слова, которые есть только в одном из них. Высокий, если одно название — это другое плюс лишние слова.

- **Мера Жаккара по словам** — доля общих слов среди всех слов двух названий:
  J(A, B) = |A ∩ B| / |A ∪ B|, где A и B — множества слов первого и второго названия.
  Принимает значения от 0 (нет общих слов) до 1 (одинаковые наборы слов); не зависит от порядка слов и повторов.


In [72]:
dict_fuzz = {'fz_ratio': fuzz.ratio, 'fz_partial_ratio': fuzz.partial_ratio, 'fz_token_sort_ratio':  fuzz.token_sort_ratio, 'fz_token_set_ratio': fuzz.token_set_ratio}

for key, value in dict_fuzz.items():
    column = [value(name1, name2) for name1, name2 in zip(data['name_1_clean'], data['name_2_clean'])]
    data[key] = column


def jaccard(str1, str2):
    words1 = set(str1.split())
    words2 = set(str2.split())
    union = words1 | words2
    if not union:
        return 0.0
    return len(words1 & words2) / len(union)

data['jaccard'] = [jaccard(name1, name2) for name1, name2 in zip(data['name_1_clean'], data['name_2_clean'])]


In [73]:
data[['name_1_clean', 'name_2_clean', 'target', 'fz_ratio', 'fz_partial_ratio',
      'fz_token_sort_ratio', 'fz_token_set_ratio', 'jaccard']].head()

,name_1_clean,name_2_clean,target,fz_ratio,fz_partial_ratio,fz_token_sort_ratio,fz_token_set_ratio,jaccard
0,очки с футляром-змейка fedrov мод. 1047 цвет 3...,фотохромные очки с футляром-змейка fedrov мод....,0.0,78.301887,80.000000,78.301887,87.179487,0.545455
1,"зубная щетка r.o.c.s. black edition classic, ч...","зубная щетка r.o.c.s. red edition classic, сре...",0.0,80.733945,84.848485,78.899083,86.363636,0.555556
2,сандалии fre gamo,fre gamo / сандалии,0.0,44.444444,64.000000,94.444444,100.000000,0.750000
3,acuvue астигматические линзы oasys for astigma...,acuvue / контактные линзы acuvue oasys for ast...,0.0,57.435897,60.540541,55.384615,61.702128,0.250000
4,"дворцовый пуэр (5 лет), 500 г",чайная линия ореховый пуэр (500 гр) вес: 500 гр,0.0,57.894737,68.965517,52.631579,55.263158,0.153846


In [74]:
features = ['fz_ratio', 'fz_partial_ratio', 'fz_token_sort_ratio', 'fz_token_set_ratio', 'jaccard']
dict_auc = {}
for feature in features:
    dict_auc[feature] = roc_auc_score(data['target'], data[feature])

pd.Series(dict_auc).sort_values(ascending=False)

fz_partial_ratio       0.579209
fz_token_set_ratio     0.571238
fz_ratio               0.569702
fz_token_sort_ratio    0.562561
jaccard                0.560022
dtype: float64

In [75]:
data.groupby('target')[features].mean()

,fz_ratio,fz_partial_ratio,fz_token_sort_ratio,fz_token_set_ratio,jaccard
target,,,,,
0.0,56.118032,68.482888,60.772864,70.984710,0.313916
1.0,60.986490,73.742555,64.964002,76.078229,0.362735


### Вывод

| Признак | ROC-AUC | Среднее у недублей | Среднее у дублей |
|---|---|---|---|
| `fz_partial_ratio` | 0.579 | 68.5 | 73.7 |
| `fz_token_set_ratio` | 0.571 | 71.0 | 76.1 |
| `fz_ratio` | 0.570 | 56.1 | 61.0 |
| `fz_token_sort_ratio` | 0.563 | 60.8 | 65.0 |
| `jaccard` | 0.560 | 0.31 | 0.36 |

Метрики похожести названий по отдельности разделяют плохо дубли и недубли  (AUC 0.56–0.58)

In [76]:
data[['id1', 'id2', 'target'] + features].to_parquet('data/features.parquet')